In [15]:
import itertools

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
import yfinance as yf

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

ENTRY_TIMING_METRICS = (
    "rsi14",
    "bollinger_percent_b_20d",
    "normalized_z_score_20d",
    "ma_trend_10_50",
    "golden_cross",
    "ma_trend_10_50_change_10d",
    "macd_histogram_12_26_9",
    "macd_histogram_12_26_9_change_10d",
)

REGIME_METRICS = (
    "factor_spread_vs_{benchmark}",
    "volatility_annualized",
    "sortino_ratio",
    "beta_vs_{benchmark}",
    "alpha_vs_{benchmark}_annualized",
    "r_squared_vs_{benchmark}",
)

ENTRY_HISTORY_PERIOD = "2y"


def download_close_prices(tickers, period="2y"):
    """Download adjusted close prices for a list of tickers."""
    tickers = sorted({ticker.upper() for ticker in tickers})
    data = yf.download(
        tickers,
        period=period,
        auto_adjust=True,
        progress=False,
        group_by="column",
        threads=True,
    )

    if data.empty:
        raise ValueError("No price data returned by yfinance.")

    close = data["Close"] if isinstance(data.columns, pd.MultiIndex) else data
    if isinstance(close, pd.Series):
        close = close.to_frame()

    close = close.sort_index().dropna(how="all")
    return close


def normalize_prices(prices, base=100.0):
    """Normalize each series to a shared base value using its first valid observation."""
    first_valid = prices.apply(lambda col: col.dropna().iloc[0] if not col.dropna().empty else np.nan)
    return prices.divide(first_valid, axis=1) * base


def equal_weight_factor_index(prices, base=100.0):
    """Create an equal-weighted composite index from a constituent price frame."""
    clean = prices.dropna(how="any")
    if clean.empty:
        raise ValueError("Not enough overlapping data to build the factor index.")
    normalized = normalize_prices(clean, base=base)
    index = normalized.mean(axis=1)
    index.name = "factor_index"
    return index


def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))


def compute_bollinger_percent_b(series, window=20, num_std=2):
    rolling = series.rolling(window)
    middle = rolling.mean()
    std = rolling.std()
    upper = middle + num_std * std
    lower = middle - num_std * std
    band_width = upper - lower
    percent_b = (series - lower) / band_width.replace(0, np.nan)
    return percent_b


def compute_macd_histogram(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line - signal_line


def compute_beta_alpha_r2(factor_returns, benchmark_returns):
    aligned = pd.concat([factor_returns, benchmark_returns], axis=1, join="inner").dropna()
    if len(aligned) < 2:
        return np.nan, np.nan, np.nan

    y = aligned.iloc[:, 0].values
    x = aligned.iloc[:, 1].values
    beta = np.cov(x, y, ddof=1)[0, 1] / np.var(x, ddof=1) if np.var(x, ddof=1) > 0 else np.nan
    alpha_daily = np.mean(y - beta * x) if np.isfinite(beta) else np.nan
    alpha_annualized = alpha_daily * 252 if np.isfinite(alpha_daily) else np.nan
    correlation = np.corrcoef(x, y)[0, 1] if len(aligned) > 1 else np.nan
    r_squared = correlation**2 if np.isfinite(correlation) else np.nan
    return beta, alpha_annualized, r_squared


def compute_sortino_ratio(returns, target=0.0):
    excess_returns = returns.dropna() - target
    downside_returns = excess_returns[excess_returns < 0]
    downside_deviation = downside_returns.std(ddof=1) * np.sqrt(252) if len(downside_returns) > 1 else np.nan
    annualized_return = excess_returns.mean() * 252 if not excess_returns.empty else np.nan
    if not np.isfinite(downside_deviation) or downside_deviation == 0:
        return np.nan
    return annualized_return / downside_deviation


def compute_entry_timing_metrics(factor_index, constituent_prices):
    latest = factor_index.dropna().iloc[-1]
    rolling_20 = factor_index.rolling(20)
    middle = rolling_20.mean().iloc[-1]
    std = rolling_20.std().iloc[-1]

    upper = middle + 2 * std if pd.notna(std) else np.nan
    lower = middle - 2 * std if pd.notna(std) else np.nan
    band_width = upper - lower
    bollinger_percent_b = (latest - lower) / band_width if pd.notna(band_width) and band_width != 0 else np.nan
    z_score = (latest - middle) / std if pd.notna(std) and std != 0 else np.nan

    rsi14 = compute_rsi(factor_index, period=14).iloc[-1]
    ma_10 = factor_index.rolling(10).mean().iloc[-1]
    ma_50 = factor_index.rolling(50).mean().iloc[-1]
    ma_200 = factor_index.rolling(200).mean().iloc[-1]
    ma_trend_10_50 = (ma_10 / ma_50) - 1 if pd.notna(ma_10) and pd.notna(ma_50) and ma_50 != 0 else np.nan
    ma_trend_series_10_50 = factor_index.rolling(10).mean().div(factor_index.rolling(50).mean()).sub(1)
    ma_trend_10_50_change_10d = ma_trend_series_10_50.iloc[-1] - ma_trend_series_10_50.shift(10).iloc[-1] if len(ma_trend_series_10_50.dropna()) > 10 else np.nan
    golden_cross = bool(ma_50 > ma_200) if pd.notna(ma_50) and pd.notna(ma_200) else False
    macd_histogram_12_26_9 = compute_macd_histogram(factor_index).iloc[-1]
    macd_histogram_series_12_26_9 = compute_macd_histogram(factor_index)
    macd_histogram_12_26_9_change_10d = macd_histogram_series_12_26_9.iloc[-1] - macd_histogram_series_12_26_9.shift(10).iloc[-1] if len(macd_histogram_series_12_26_9.dropna()) > 10 else np.nan

    pct_above_50dma = (constituent_prices.iloc[-1] > constituent_prices.rolling(50).mean().iloc[-1]).mean() * 100

    return {
        "rsi14": rsi14,
        "bollinger_percent_b_20d": bollinger_percent_b,
        "normalized_z_score_20d": z_score,
        "ma_trend_10_50": ma_trend_10_50,
        "golden_cross": golden_cross,
        "ma_trend_10_50_change_10d": ma_trend_10_50_change_10d,
        "macd_histogram_12_26_9": macd_histogram_12_26_9,
        "macd_histogram_12_26_9_change_10d": macd_histogram_12_26_9_change_10d,
        "pct_etfs_above_50dma": pct_above_50dma,
    }


def compute_regime_metrics(factor_index, benchmark_index, benchmark_name):
    factor_returns = factor_index.pct_change()
    benchmark_returns = benchmark_index.pct_change()
    beta, alpha, r_squared = compute_beta_alpha_r2(factor_returns, benchmark_returns)

    benchmark_key = benchmark_name.lower()
    latest = factor_index.dropna().iloc[-1]
    benchmark_latest = benchmark_index.dropna().iloc[-1]
    spread = latest / benchmark_latest
    volatility_annualized = factor_returns.std(ddof=1) * np.sqrt(252)
    sortino_ratio = compute_sortino_ratio(factor_returns)

    return {
        f"factor_spread_vs_{benchmark_key}": spread,
        "volatility_annualized": volatility_annualized,
        "sortino_ratio": sortino_ratio,
        f"beta_vs_{benchmark_key}": beta,
        f"alpha_vs_{benchmark_key}_annualized": alpha,
        f"r_squared_vs_{benchmark_key}": r_squared,
    }


def compute_factor_metrics(factor_index, benchmark_index, constituent_prices, benchmark_name):
    entry_metrics = compute_entry_timing_metrics(factor_index, constituent_prices)
    regime_metrics = compute_regime_metrics(factor_index, benchmark_index, benchmark_name)
    return {**entry_metrics, **regime_metrics}


def split_metric_tables(metrics_df, benchmark_name):
    benchmark_key = benchmark_name.lower()
    entry_columns = list(ENTRY_TIMING_METRICS) + ["pct_etfs_above_50dma"]
    regime_columns = [
        metric.format(benchmark=benchmark_key)
        for metric in REGIME_METRICS
    ]

    metadata_columns = ["constituents", "constituent_count"]
    entry_metrics_df = metrics_df[metadata_columns + entry_columns].copy() if not metrics_df.empty else pd.DataFrame()
    regime_metrics_df = metrics_df[metadata_columns + regime_columns].copy() if not metrics_df.empty else pd.DataFrame()

    return entry_metrics_df, regime_metrics_df


def build_factor_dashboard(factor_groups, benchmark="SPY", period="2y"):
    """Download data, build factor composites, and compute metrics for each factor group."""
    benchmark_name = benchmark.upper()
    factor_groups = {name: sorted({ticker.upper() for ticker in tickers}) for name, tickers in factor_groups.items()}
    all_tickers = sorted({benchmark_name, *itertools.chain.from_iterable(factor_groups.values())})
    close = download_close_prices(all_tickers, period=period)
    entry_history = download_close_prices(all_tickers, period=ENTRY_HISTORY_PERIOD)

    if benchmark_name not in close.columns:
        raise KeyError(f"Benchmark {benchmark_name!r} was not returned by yfinance.")

    benchmark_index = close[benchmark_name].dropna()
    factor_series = {}
    metric_rows = []

    for factor_name, tickers in factor_groups.items():
        available = [ticker for ticker in tickers if ticker in close.columns]
        history_available = [ticker for ticker in tickers if ticker in entry_history.columns]
        if len(available) < 1 or len(history_available) < 1:
            continue

        subset = close[available + [benchmark_name]].dropna(how="any")
        history_subset = entry_history[history_available + [benchmark_name]].dropna(how="any")
        if subset.empty or history_subset.empty:
            continue

        factor_prices = subset[available]
        factor_index = equal_weight_factor_index(factor_prices, base=100.0)
        factor_series[factor_name] = factor_index

        history_factor_prices = history_subset[history_available]
        history_factor_index = equal_weight_factor_index(history_factor_prices, base=100.0)
        history_benchmark_slice = history_subset[benchmark_name].loc[history_factor_index.index]

        metrics = compute_factor_metrics(
            history_factor_index,
            history_benchmark_slice,
            history_factor_prices.loc[history_factor_index.index],
            benchmark_name,
        )
        metrics["factor"] = factor_name
        metrics["constituents"] = ", ".join(available)
        metrics["constituent_count"] = len(available)
        metric_rows.append(metrics)

    metrics_df = pd.DataFrame(metric_rows).set_index("factor") if metric_rows else pd.DataFrame()
    metrics_df = metrics_df.sort_values(f"factor_spread_vs_{benchmark_name.lower()}", ascending=False) if not metrics_df.empty else metrics_df
    entry_metrics_df, regime_metrics_df = split_metric_tables(metrics_df, benchmark_name)
    return close, factor_series, benchmark_index, entry_metrics_df, regime_metrics_df


def build_period_figure(factor_series, benchmark_index, benchmark_name, period="2y"):
    figure = go.Figure()

    benchmark_slice = benchmark_index.dropna()
    benchmark_slice = benchmark_slice / benchmark_slice.iloc[0] * 100
    figure.add_trace(
        go.Scatter(
            x=benchmark_slice.index,
            y=benchmark_slice.values,
            mode="lines",
            name=benchmark_name,
            line=dict(color="#111827", width=2, dash="dash"),
        )
)

    colors = ["#0F172A", "#2563EB", "#16A34A", "#D97706", "#DC2626", "#7C3AED", "#0891B2"]

    for i, (factor_name, series) in enumerate(factor_series.items()):
        slice_ = series.dropna()
        slice_ = slice_ / slice_.iloc[0] * 100
        figure.add_trace(
            go.Scatter(
                x=slice_.index,
                y=slice_.values,
                mode="lines",
                name=factor_name,
                line=dict(color=colors[i % len(colors)], width=2),
            )
)

    figure.update_layout(
        template="plotly_white",
        height=550,
        width=1400,
        title=f"Factor composites vs {benchmark_name} over {period}",
        legend_title_text="Series",
        hovermode="x unified",
        margin=dict(l=30, r=30, t=70, b=30),
    )
    return figure


# Replace this dictionary with your own factor groups.
factor_groups = {
    "Value": ["VTV", "AVLV", "RPV", 'AVUV', 'VLUE'],
    "Momentum": ["FMTM", "SPMO", "QMOM", "VFMO", "PDP"],
    "Quality": ["QUAL", "SPHQ", "VIG", "COWZ", "JQUA"],
    "Stability": ["SPLV", "USMV", "FDLO", "BTAL", "SPD"],
    "Sentiment": ["PKW", "BUYB", "BUZZ", 'SGRT', 'NEWZ'],
    "Dividend": ["SCHD", "SDY", "DGRO", "VYM", "CGDV"],
    "Market": ["SPY", "QQQ", "IWM", "DIA", "VTI"],
    "Singleton": ["VTV"],
}

benchmark_name = "QLD"
period = "30d"

close_prices, factor_series, benchmark_index, entry_metrics_df, regime_metrics_df = build_factor_dashboard(
    factor_groups=factor_groups,
    benchmark=benchmark_name,
    period=period,
)

period_figure = build_period_figure(factor_series, benchmark_index, benchmark_name, period=period)

print("Entry timing metrics: indicators anchored to the latest bar and computed from longer history\n")
display(entry_metrics_df.round(4))
print("Regime metrics: factor behavior versus the benchmark over the selected download period\n")
display(regime_metrics_df.round(4))
display(HTML(period_figure.to_html(include_plotlyjs="cdn", full_html=False)))


Entry timing metrics: indicators anchored to the latest bar and computed from longer history



,constituents,constituent_count,rsi14,bollinger_percent_b_20d,normalized_z_score_20d,ma_trend_10_50,golden_cross,ma_trend_10_50_change_10d,macd_histogram_12_26_9,macd_histogram_12_26_9_change_10d,pct_etfs_above_50dma
factor,,,,,,,,,,,
Value,"AVLV, AVUV, RPV, VLUE, VTV",5,54.7292,0.5135,0.0539,0.0359,True,-0.0121,-0.3728,-0.1555,100.0
Momentum,"FMTM, PDP, QMOM, SPMO, VFMO",5,46.2394,0.2712,-0.9153,0.0421,True,0.0034,-0.5732,-0.5852,40.0
Market,"DIA, IWM, QQQ, SPY, VTI",5,55.5800,0.6632,0.6529,0.0215,True,-0.0062,-0.0725,0.1631,100.0
Singleton,VTV,1,64.7014,0.8119,1.2475,0.0346,True,0.0025,-0.0504,-0.1854,100.0
Dividend,"CGDV, DGRO, SCHD, SDY, VYM",5,67.4499,1.0647,2.2588,0.0148,True,-0.0052,0.0662,0.1439,100.0
Quality,"COWZ, JQUA, QUAL, SPHQ, VIG",5,61.1356,0.8293,1.3173,0.0198,True,-0.0083,0.0229,0.1604,100.0
Stability,"BTAL, FDLO, SPD, SPLV, USMV",5,68.0617,1.1797,2.7187,-0.0012,True,-0.0031,0.2275,0.3286,80.0
Sentiment,"BUYB, BUZZ, NEWZ, PKW, SGRT",5,53.2362,0.4840,-0.0639,NaN,False,NaN,-0.0317,-0.0021,0.0


Regime metrics: factor behavior versus the benchmark over the selected download period



,constituents,constituent_count,factor_spread_vs_qld,volatility_annualized,sortino_ratio,beta_vs_qld,alpha_vs_qld_annualized,r_squared_vs_qld
factor,,,,,,,,
Value,"AVLV, AVUV, RPV, VLUE, VTV",5,1.7116,0.1637,1.9992,0.2718,0.1327,0.5394
Momentum,"FMTM, PDP, QMOM, SPMO, VFMO",5,1.6454,0.2387,1.8707,0.4607,0.0523,0.7690
Market,"DIA, IWM, QQQ, SPY, VTI",5,1.5755,0.1776,1.4856,0.3759,0.0542,0.8762
Singleton,VTV,1,1.5736,0.1340,1.9114,0.2088,0.1095,0.4753
Dividend,"CGDV, DGRO, SCHD, SDY, VYM",5,1.5407,0.1278,1.8656,0.1955,0.1030,0.4577
Quality,"COWZ, JQUA, QUAL, SPHQ, VIG",5,1.4690,0.1449,1.4368,0.2819,0.0489,0.7409
Stability,"BTAL, FDLO, SPD, SPLV, USMV",5,1.2392,0.0859,1.1531,0.0925,0.0274,0.2271
Sentiment,"BUYB, BUZZ, NEWZ, PKW, SGRT",5,1.1356,0.1996,1.3884,0.3410,0.0831,0.8565
